In [6]:
import pandas as pd
import numpy as np
from scipy import stats

file_path = 'danniye.xlsx'
df_audience = pd.read_excel(file_path, sheet_name='Данные об аудитории')
df_ab_tests = pd.read_excel(file_path, sheet_name='Данные АБ тестов')
df_listers = pd.read_excel(file_path, sheet_name='Листеры')

df_audience['date'] = pd.to_datetime(df_audience['date'])

In [7]:
mau = df_audience['user_id'].nunique()
print(f"1. MAU: {mau}")

1. MAU: 7639


In [8]:
dau_avg = round(df_audience.groupby('date')['user_id'].nunique().mean())
print(f"2. DAU: {dau_avg}")

2. DAU: 560


In [9]:
users_with_views = df_audience[df_audience['view_adverts'] > 0]['user_id'].nunique()
conv = (users_with_views / mau) * 100
print(f"5. Конверсия в просмотр: {conv:.1f}%")

5. Конверсия в просмотр: 46.3%


In [10]:
avg_views = df_audience.groupby('user_id')['view_adverts'].sum().mean()
print(f"6. Среднее кол-во просмотров: {avg_views:.1f}")

6. Среднее кол-во просмотров: 2.9


In [11]:
#Retention 1-го дня (когорта 1 ноября)
cohort = df_audience[df_audience['date'] == '2023-11-01']['user_id'].unique()
returned = df_audience[df_audience['date'] == '2023-11-02']['user_id'].unique()
retention = (len(np.intersect1d(cohort, returned)) / len(cohort)) * 100
print(f"3. Retention 1-го дня: {retention:.1f}%")

3. Retention 1-го дня: 26.6%


In [12]:
for i in range(1, 4):
    test_grp = df_ab_tests[(df_ab_tests['experiment_num'] == i) & (df_ab_tests['experiment_group'] == 'test')]['revenue']
    ctrl_grp = df_ab_tests[(df_ab_tests['experiment_num'] == i) & (df_ab_tests['experiment_group'] == 'control')]['revenue']
    
    p_val = stats.mannwhitneyu(test_grp, ctrl_grp).pvalue
    print(f"Тест №{i}: p-value = {p_val:.4f}")
    print(f"  Результат: {'Значимо' if p_val < 0.05 else 'Незначимо'}")

Тест №1: p-value = 0.7960
  Результат: Незначимо
Тест №2: p-value = 0.0085
  Результат: Значимо
Тест №3: p-value = 0.0010
  Результат: Значимо


In [13]:
print(f"9. Средний доход (Листеры): {df_listers['revenue'].mean():.1f}")
print(f"10. Медиана возраста: {df_listers['age'].median()}")

9. Средний доход (Листеры): 30.7
10. Медиана возраста: 28.0
